In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import time

In [21]:
data=pd.read_csv('/opt/neurothink/neuro-deb05933716f4eb6be138022abf35d2f/train.csv')
test=pd.read_csv('/opt/neurothink/neuro-1d4c617f8ef0447bafed400cb9e1ff6b/test.csv')

In [26]:
data.head()
#len(test)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [31]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [27]:
data['Age'].fillna(data['Age'].median(),inplace=True)
data.drop('Cabin',axis=1,inplace=True)
data['Embarked'].fillna('S',inplace=True)
data.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)
data['Pclass']=data['Pclass'].astype(str)
data['Accompanied']=0
for i in range(data.shape[0]):
    if (data.loc[i,'SibSp']+data.loc[i,'Parch'])>0:
        data.loc[i,'Accompanied']=1
data['Accompanied']=data['Accompanied'].astype(str)
data.drop(['SibSp','Parch'],axis=1,inplace=True)
data=pd.get_dummies(data)

In [28]:
data

,Survived,Age,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Accompanied_0,Accompanied_1
0,0,22.0,7.2500,0,0,1,0,1,0,0,1,0,1
1,1,38.0,71.2833,1,0,0,1,0,1,0,0,0,1
2,1,26.0,7.9250,0,0,1,1,0,0,0,1,1,0
3,1,35.0,53.1000,1,0,0,1,0,0,0,1,0,1
4,0,35.0,8.0500,0,0,1,0,1,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,27.0,13.0000,0,1,0,0,1,0,0,1,1,0
887,1,19.0,30.0000,1,0,0,1,0,0,0,1,1,0
888,0,28.0,23.4500,0,0,1,1,0,0,0,1,0,1
889,1,26.0,30.0000,1,0,0,0,1,1,0,0,1,0


In [29]:
X=data[['Age','Fare','Pclass_1','Pclass_2','Pclass_3','Sex_female','Sex_male','Embarked_C','Embarked_Q','Embarked_S','Accompanied_0','Accompanied_1']]
y=data['Survived']

In [30]:
test['Fare'].fillna(test['Fare'].median(),inplace=True)
test['Age'].fillna(data['Age'].median(),inplace=True)
test.drop('Cabin',axis=1,inplace=True)
test['Embarked'].fillna('S',inplace=True)
test.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)
test['Pclass']=test['Pclass'].astype(str)
test['Accompanied']=0
for i in range(test.shape[0]):
    if (test.loc[i,'SibSp']+test.loc[i,'Parch'])>0:
        test.loc[i,'Accompanied']=1
test['Accompanied']=test['Accompanied'].astype(str)
test.drop(['SibSp','Parch'],axis=1,inplace=True)
test=pd.get_dummies(test)

In [31]:
from sklearn.model_selection import GridSearchCV
from xgboost.sklearn import XGBClassifier
import lightgbm as lgb
from sklearn.ensemble import GradientBoostingClassifier as gbr


In [32]:
xgb=XGBClassifier(learning_rate=0.1,n_estimators=500,max_depth=5,min_child_weight=1,gamma=0,
subsample=0.8,colsample_bytree=0.8,objective= 'binary:logistic',seed=0)

In [12]:
start = time.time_ns()
param_test = {'n_estimators':range(30,40,50)}
gs = GridSearchCV(estimator = xgb,param_grid = param_test, scoring='roc_auc',n_jobs=-1,cv=5)
gs.fit(X,y)
print(gs.best_params_,gs.best_score_)
print(gs.cv_results_['mean_test_score'])
end = time.time_ns()
print('**************')
print('TIME USED (ms):', (end-start)/1000)

/opt/conda/lib/python3.9/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
/opt/conda/lib/python3.9/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)
/opt/conda/lib/python3.9/site-packages/xgboost/data.py:262: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  elif isinstance(data.columns, (pd.Int64Index, pd.RangeIndex)):
/opt/conda/lib

[03:52:56] WARNING: ../src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
{'n_estimators': 30} 0.8684815528996529
[0.86848155]
**************
TIME USED (ms): 476564962.324


In [47]:
#!pip install catboost

     |████████████████████████████████| 76.2 MB 219 kB/s             


In [33]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(eval_metric='Accuracy', use_best_model=True,  random_seed=42)


In [34]:
param_test = {'iterations':[30,40,50], 'use_best_model': [False]}
"""
params = {'depth':[3,1,2,6,4,5,7,8,9,10],
          'iterations':[250,100,500,1000],
          'learning_rate':[0.03,0.001,0.01,0.1,0.2,0.3], 
          'l2_leaf_reg':[3,1,5,10,100],
          'border_count':[32,5,10,20,50,100,200],
          'ctr_border_count':[50,5,10,20,100,200],
          'thread_count':4}
"""
start = time.time_ns()
gs_cat = GridSearchCV(estimator = cat, param_grid = param_test, scoring='roc_auc',n_jobs=-1,cv=5)
gs_cat.fit(X,y)
print(gs_cat.best_params_,gs_cat.best_score_)
print(gs_cat.cv_results_['mean_test_score'])
end = time.time_ns()
print('**************')
print('TIME USED (ms):', (end-start)/1000)

Learning rate set to 0.222169
0:	learn: 0.7826087	total: 47.3ms	remaining: 1.37s
1:	learn: 0.8106592	total: 48ms	remaining: 672ms
2:	learn: 0.8204769	total: 49.3ms	remaining: 444ms
3:	learn: 0.8288920	total: 50ms	remaining: 325ms
4:	learn: 0.8274895	total: 50.7ms	remaining: 253ms
5:	learn: 0.8260870	total: 51.3ms	remaining: 205ms
6:	learn: 0.8302945	total: 52ms	remaining: 171ms
7:	learn: 0.8373072	total: 52.9ms	remaining: 146ms
8:	learn: 0.8302945	total: 53.2ms	remaining: 124ms
9:	learn: 0.8274895	total: 53.9ms	remaining: 108ms
10:	learn: 0.8330996	total: 54.6ms	remaining: 94.3ms
11:	learn: 0.8373072	total: 55.3ms	remaining: 82.9ms
12:	learn: 0.8345021	total: 55.7ms	remaining: 72.8ms
13:	learn: 0.8345021	total: 56.3ms	remaining: 64.4ms
14:	learn: 0.8330996	total: 57.8ms	remaining: 57.8ms
15:	learn: 0.8359046	total: 58.4ms	remaining: 51.1ms
16:	learn: 0.8330996	total: 59.1ms	remaining: 45.2ms
17:	learn: 0.8401122	total: 59.7ms	remaining: 39.8ms
18:	learn: 0.8429173	total: 61.1ms	remaini

In [35]:
orig_data=pd.read_csv('/opt/neurothink/neuro-deb05933716f4eb6be138022abf35d2f/train.csv')
orig_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [36]:
for col in orig_data:
    orig_data[col] = orig_data[col].fillna('NaN')
train_X = orig_data.loc[:, orig_data.columns!='Survived']
train_Y = orig_data['Survived']
train_X.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [37]:
categorical_indices = [0,1,2,3,5,6,7,9,10]
start = time.time_ns()
gs_cat = GridSearchCV(estimator = cat, param_grid = param_test, scoring='roc_auc',n_jobs=-1,cv=5)
gs_cat.fit(train_X,train_Y, cat_features=categorical_indices)
end = time.time_ns()
print(gs_cat.best_params_,gs_cat.best_score_)
print(gs_cat.cv_results_['mean_test_score'])
print('**************')
print('TIME USED (ms):', (end-start)/1000)

Learning rate set to 0.222169
0:	learn: 0.8274895	total: 47.7ms	remaining: 1.38s
1:	learn: 0.8162693	total: 48.7ms	remaining: 682ms
2:	learn: 0.8106592	total: 49.3ms	remaining: 443ms
3:	learn: 0.8274895	total: 49.8ms	remaining: 324ms
4:	learn: 0.8288920	total: 50.7ms	remaining: 253ms
5:	learn: 0.8302945	total: 51.3ms	remaining: 205ms
6:	learn: 0.8316971	total: 51.8ms	remaining: 170ms
7:	learn: 0.8373072	total: 52.4ms	remaining: 144ms
8:	learn: 0.8330996	total: 52.6ms	remaining: 123ms
9:	learn: 0.8359046	total: 53.2ms	remaining: 106ms
10:	learn: 0.8359046	total: 53.7ms	remaining: 92.7ms
11:	learn: 0.8429173	total: 54.2ms	remaining: 81.3ms
12:	learn: 0.8429173	total: 55ms	remaining: 71.9ms
13:	learn: 0.8457223	total: 55.5ms	remaining: 63.4ms
14:	learn: 0.8457223	total: 55.7ms	remaining: 55.7ms
15:	learn: 0.8457223	total: 56.2ms	remaining: 49.2ms
16:	learn: 0.8429173	total: 56.8ms	remaining: 43.4ms
17:	learn: 0.8457223	total: 57.3ms	remaining: 38.2ms
18:	learn: 0.8443198	total: 57.9ms	rem

In [38]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(max_depth=2, random_state=0)

In [40]:
start = time.time_ns()
param_test = {"max_depth":[2,3,4,8]}
gs_rf = GridSearchCV(estimator = rf, param_grid = param_test, scoring='roc_auc',n_jobs=-1,cv=5)
gs_rf.fit(X,y)
end = time.time_ns()
print(gs_rf.best_params_,gs_rf.best_score_)
print(gs_rf.cv_results_['mean_test_score'])
print('**************')
print('TIME USED (ms):', (end-start)/1000)

{'max_depth': 8} 0.8691885270262872
[0.84435619 0.85402126 0.85756261 0.86918853]
**************
TIME USED (ms): 651655.501
3:	learn: 0.8078541	total: 4.88ms	remaining: 31.7ms
4:	learn: 0.8078541	total: 6.93ms	remaining: 34.6ms
5:	learn: 0.8162693	total: 7.7ms	remaining: 30.8ms
6:	learn: 0.8148668	total: 8.18ms	remaining: 26.9ms
7:	learn: 0.8232819	total: 8.86ms	remaining: 24.4ms
8:	learn: 0.8288920	total: 10.4ms	remaining: 24.2ms
9:	learn: 0.8302945	total: 11.1ms	remaining: 22.2ms
10:	learn: 0.8345021	total: 12.1ms	remaining: 20.8ms
11:	learn: 0.8302945	total: 12.7ms	remaining: 19ms
12:	learn: 0.8302945	total: 13.3ms	remaining: 17.4ms
13:	learn: 0.8345021	total: 13.9ms	remaining: 15.9ms
14:	learn: 0.8330996	total: 14.6ms	remaining: 14.6ms
15:	learn: 0.8330996	total: 15.2ms	remaining: 13.3ms
16:	learn: 0.8359046	total: 15.8ms	remaining: 12.1ms
17:	learn: 0.8373072	total: 16.4ms	remaining: 10.9ms
18:	learn: 0.8429173	total: 16.9ms	remaining: 9.76ms
19:	learn: 0.8457223	total: 17.2ms	rem